# INT8 Quantization from Scratch — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Post-training quantization (PTQ) is how most large models are served: int8 weights halve
memory and HBM traffic relative to bf16 (int4 quarters it), and int8 tensor cores run about
twice the bf16 FLOP rate. PyTorch's `torch.ao.quantization`, TensorRT, LLM.int8(), SmoothQuant,
GPTQ and AWQ all build on the same **affine map** between a real value $x$ and an integer code $q$:

$$\hat x = s\,(q - z), \qquad q = \operatorname{clamp}\!\big(\operatorname{round}(x / s) + z,\; q_{\min},\; q_{\max}\big)$$

with a positive **scale** $s$ and an integer **zero point** $z$. Rounding to the nearest code
bounds the error by half a step: $|\hat x - x| \le s/2$ for every $x$ inside the representable
range.

The payoff is that a linear layer runs on integers. With activations $\hat x = s_x (q^x - z_x)$
(per-tensor) and weights $\hat W_{j\cdot} = s_{w,j}\, q^w_{j\cdot}$ (symmetric, one scale per output
channel $j$):

$$y_{ij} = \sum_k \hat x_{ik}\,\hat W_{jk} = s_x\, s_{w,j} \underbrace{\sum_k \big(q^x_{ik} - z_x\big)\, q^w_{jk}}_{\text{exact integer accumulator (int32)}}$$

— all the multiply-adds happen in integer arithmetic and the only floating-point work is a
single rescale per output.

---

### Requirements

1. **`quantize_symmetric(x, num_bits=8, axis=None) -> (q, scale)`**
   - $q_{\max} = 2^{b-1} - 1$ (127 for 8 bits). Codes live in $[-q_{\max}, q_{\max}]$ — the code
     $-2^{b-1}$ is left unused so the grid is symmetric — and the zero point is 0.
   - `axis=None` (per-tensor): $s = \max|x| \,/\, q_{\max}$, returned as a 0-dim float tensor.
   - `axis=k` (per-channel, `k` may be negative): one scale per index of dimension `k`, computed
     from the max of $|x|$ over **all other** dimensions; returned as a 1-D tensor of shape
     `(x.shape[k],)`. For an `nn.Linear` weight `(out_features, in_features)`, `axis=0` is
     per-output-channel.
   - $q = \operatorname{clamp}(\operatorname{round}(x / s), -q_{\max}, q_{\max})$, stored as
     `torch.int8` (also for `num_bits < 8`, e.g. 4-bit codes in $[-7, 7]$).
   - An all-zero tensor or channel must still get a positive, finite scale (use 1.0) — never NaN.
2. **`quantize_asymmetric(x, num_bits=8) -> (q, scale, zero_point)`** (per-tensor)
   - Codes in $[0, 2^b - 1]$, stored as `torch.uint8`.
   - Widen the range to contain zero: $x_{\min} = \min(\min x, 0)$, $x_{\max} = \max(\max x, 0)$,
     and $s = (x_{\max} - x_{\min}) / (2^b - 1)$ (1.0 if $x_{\max} = x_{\min}$).
   - `zero_point` is an **integer** (Python `int` or 0-dim integer tensor):
     $z = \operatorname{clamp}(\operatorname{round}(-x_{\min}/s), 0, 2^b - 1)$, so that $0.0$ is
     exactly representable (it quantizes to $z$ and dequantizes to exactly 0).
   - $q = \operatorname{clamp}(\operatorname{round}(x / s) + z,\; 0,\; 2^b - 1)$.
3. **`dequantize(q, scale, zero_point=None, axis=None) -> Tensor`** — float32 $(q - z)\,s$;
   `zero_point=None` means 0. With `axis`, `scale` is 1-D and broadcasts along that dimension.
4. **`int8_linear(x_q, x_scale, x_zero_point, w_q, w_scale) -> Tensor`**
   - `x_q`: `(M, K)` uint8 activation codes, quantized per-tensor with `x_scale` and
     `x_zero_point` (`None` means 0). `w_q`: `(N, K)` int8 weight codes (the `nn.Linear`
     layout), symmetric, with `w_scale` a scalar or a `(N,)` tensor (per output channel).
   - Return the float32 `(M, N)` result $\approx \hat x\, \hat W^\top$: compute
     $\sum_k (q^x_{ik} - z_x)\, q^w_{jk}$ **exactly with integer arithmetic**, then multiply once by
     `x_scale * w_scale` (a per-channel `w_scale` scales the output columns).
   - "Exactly" is tested: with `x_scale = w_scale = 1` the output must equal the exact integer
     dot products (rounded once to the output dtype) even when they exceed $2^{24}$.

---

### Constraints

- ✅ `torch.round`, `torch.clamp`, `amax`, and integer matmuls (`int32 @ int32` works on CPU in
  recent PyTorch; int64 does too).
- ✅ Use `torch.quantize_per_tensor` / `torch.quantize_per_channel` only to check yourself.
- ❌ No `torch.ao.quantization`, `torch.quantize_per_*` or `torch._int_mm` inside your functions.
- ❌ Don't dequantize inside `int8_linear` — the matmul itself must be integer.
- ❌ Don't accumulate in float32: products reach $255 \cdot 127$, and sums over $K$ pass $2^{24}$,
  beyond which float32 cannot represent every integer.
- ❌ Don't matmul `int8 @ int8` or subtract a zero point from a `uint8` tensor directly —
  PyTorch keeps the 8-bit dtype and silently wraps around.

---

<details>
  <summary>💡 Hint</summary>

  - Casting with `.to(torch.int8)` truncates toward zero; round first.
  - For per-channel scales, think of `amax` with a *list* of dims, then reshape the `(C,)` scale
    so it broadcasts against `x` (size `C` at `axis`, 1 everywhere else).
  - Why must the asymmetric zero point be rounded to an integer *before* you quantize? What
    would happen to zero-padding or ReLU outputs otherwise?
  - What dtype does `uint8_tensor - 7` have? And `int8_tensor @ int8_tensor`? Try it.

</details>

In [ ]:
import warnings

import torch

In [ ]:
torch.manual_seed(0)

# A Linear layer's weight with one outlier output channel (a common sight in LLMs)
W = torch.randn(64, 128) * 0.02
W[7] *= 50.0

# Activations: a batch of post-ReLU features (non-negative) and a signed input
acts_relu = torch.relu(torch.randn(16, 128))
acts_signed = torch.randn(16, 128) * 1.5 + 0.3

print(f"W: {tuple(W.shape)}, max|W| = {W.abs().max():.3f}, typical |W| = {W.abs().median():.4f}")
print(f"ReLU activations in [{acts_relu.min():.2f}, {acts_relu.max():.2f}]")

In [ ]:
def quantize_symmetric(x, num_bits=8, axis=None):
    """Symmetric (zero point 0) quantization, per tensor or per channel.

    Args:
        x: float tensor of any shape.
        num_bits: code width b; codes live in [-(2^(b-1) - 1), 2^(b-1) - 1].
        axis: None for one scale, or the dimension that keeps its own scale (may be negative).

    Returns:
        q: torch.int8 tensor with x's shape.
        scale: 0-dim tensor (axis=None) or 1-D tensor of shape (x.shape[axis],).
    """
    qmax = 2 ** (num_bits - 1) - 1
    if axis is None:
        amax = x.abs().amax()
    else:
        axis = axis % x.ndim
        others = [d for d in range(x.ndim) if d != axis]
        amax = x.abs().amax(dim=others) if others else x.abs()    # (C,)
    # An all-zero tensor/channel would give scale 0 -> 0/0 = NaN; any positive scale works.
    scale = torch.where(amax > 0, amax / qmax, torch.ones_like(amax))
    if axis is None:
        scale_b = scale
    else:
        shape = [1] * x.ndim
        shape[axis] = -1
        scale_b = scale.reshape(shape)                             # broadcast along `axis`
    q = torch.clamp(torch.round(x / scale_b), -qmax, qmax).to(torch.int8)
    return q, scale

In [ ]:
def quantize_asymmetric(x, num_bits=8):
    """Asymmetric (affine) per-tensor quantization to unsigned codes.

    Args:
        x: float tensor of any shape.
        num_bits: code width b; codes live in [0, 2^b - 1].

    Returns:
        q: torch.uint8 tensor with x's shape.
        scale: 0-dim float tensor.
        zero_point: int — the code that represents 0.0 exactly.
    """
    qmax = 2 ** num_bits - 1
    x_min = torch.clamp(x.min(), max=0.0)          # the range must contain 0 ...
    x_max = torch.clamp(x.max(), min=0.0)
    span = x_max - x_min
    scale = torch.where(span > 0, span / qmax, torch.ones_like(span))
    # ... and 0 must land exactly on a code: round the zero point BEFORE quantizing.
    zero_point = int(torch.clamp(torch.round(-x_min / scale), 0, qmax))
    q = torch.clamp(torch.round(x / scale) + zero_point, 0, qmax).to(torch.uint8)
    return q, scale, zero_point

In [ ]:
def dequantize(q, scale, zero_point=None, axis=None):
    """Map integer codes back to float32: (q - zero_point) * scale.

    Args:
        q: integer tensor (int8 or uint8).
        scale: number / 0-dim tensor, or a 1-D tensor of per-channel scales along `axis`.
        zero_point: integer or None (= 0).
        axis: dimension the per-channel scale runs along, or None.

    Returns:
        float32 tensor with q's shape.
    """
    scale = torch.as_tensor(scale, dtype=torch.float32)
    if axis is not None:
        shape = [1] * q.ndim
        shape[axis] = -1                           # negative axis indexes from the end too
        scale = scale.reshape(shape)
    qf = q.to(torch.float32)                       # upcast first: uint8 - z would wrap around
    if zero_point is not None:
        qf = qf - int(zero_point)                  # small integers: exact in float32
    return qf * scale

In [ ]:
def int8_linear(x_q, x_scale, x_zero_point, w_q, w_scale):
    """y ≈ dequant(x) @ dequant(w).T with exact integer accumulation and one float rescale.

    Args:
        x_q: (M, K) uint8 activation codes (per-tensor scale x_scale, zero point x_zero_point).
        x_scale: float or 0-dim tensor.
        x_zero_point: int, 0-dim integer tensor, or None (= 0).
        w_q: (N, K) int8 symmetric weight codes (nn.Linear layout).
        w_scale: float / 0-dim tensor, or (N,) per-output-channel scales.

    Returns:
        (M, N) float32 tensor.
    """
    x_int = x_q.to(torch.int32)
    if x_zero_point is not None:
        x_int = x_int - int(x_zero_point)          # in [-255, 255]
    # int32 matmul: exact as long as K * 255 * 127 < 2^31, i.e. K < ~66k — the same int32
    # accumulator int8 tensor cores use. (int8 @ int8 would stay int8 and wrap around.)
    acc = x_int @ w_q.to(torch.int32).T            # (M, N), exact
    scale = (torch.as_tensor(x_scale, dtype=torch.float32)
             * torch.as_tensor(w_scale, dtype=torch.float32))    # scalar or (N,)
    return acc.to(torch.float32) * scale           # the only floating-point step

In [ ]:
# Test 1: symmetric per-tensor — dtype, range, error bound, agreement with PyTorch
x = torch.randn(1000) * 3
q, s = quantize_symmetric(x)
assert q.dtype == torch.int8 and q.shape == x.shape
assert q.abs().max() == 127, "the largest |x| should map to ±127"
assert torch.allclose(s, x.abs().max() / 127)
err = (dequantize(q, s) - x).abs()
assert (err <= s / 2 + 1e-6).all(), f"rounding error must be <= scale/2, got {err.max() / s:.3f}·scale"
q4, s4 = quantize_symmetric(x, num_bits=4)
assert q4.abs().max() == 7 and ((dequantize(q4, s4) - x).abs() <= s4 / 2 + 1e-6).all()
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ref = torch.quantize_per_tensor(x, float(s), 0, torch.qint8).int_repr()
    assert (ref.int() - q.int()).abs().max() <= 1 and (ref != q).float().mean() < 1e-3
    print("matches torch.quantize_per_tensor")
except (AttributeError, RuntimeError, NotImplementedError):
    print("(torch.quantize_per_tensor unavailable here — skipped the comparison)")
print(f"Test 1 passed: scale = {float(s):.4f}, max error = {err.max():.4f} <= scale/2")

In [ ]:
# Test 2: per-channel vs per-tensor on the weight with an outlier channel
q_t, s_t = quantize_symmetric(W)                 # one scale
q_c, s_c = quantize_symmetric(W, axis=0)         # one scale per output channel
assert s_c.shape == (W.shape[0],)
assert torch.allclose(s_c, W.abs().amax(dim=1) / 127)
mse_t = (dequantize(q_t, s_t) - W).pow(2).mean(dim=1)
mse_c = (dequantize(q_c, s_c, axis=0) - W).pow(2).mean(dim=1)
assert (mse_c <= mse_t + 1e-12).all(), "per-channel must never be worse than per-tensor"
normal = torch.arange(W.shape[0]) != 7
print(f"MSE on the 63 normal channels: per-tensor {mse_t[normal].mean():.2e}, "
      f"per-channel {mse_c[normal].mean():.2e} ({mse_t[normal].mean() / mse_c[normal].mean():.0f}x better)")
print(f"fraction of normal weights rounded to 0 per-tensor: {(q_t[normal] == 0).float().mean():.0%}")
assert mse_t[normal].mean() > 100 * mse_c[normal].mean()
q_ax1, s_ax1 = quantize_symmetric(W, axis=-1)    # per input column, negative axis
assert s_ax1.shape == (W.shape[1],)
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ref = torch.quantize_per_channel(W, s_c.double(), torch.zeros(64, dtype=torch.long),
                                         0, torch.qint8).int_repr()
    assert (ref.int() - q_c.int()).abs().max() <= 1
    print("matches torch.quantize_per_channel")
except (AttributeError, RuntimeError, NotImplementedError):
    print("(torch.quantize_per_channel unavailable here — skipped the comparison)")
z = torch.zeros(4, 6); z[1] = 1.0
qz, sz = quantize_symmetric(z, axis=0)
assert torch.isfinite(sz).all() and (sz > 0).all(), "all-zero channels need a positive finite scale"
print("Test 2 passed")

In [ ]:
# Test 3: asymmetric — uint8 codes, integer zero point, 0.0 round-trips exactly
xa = acts_signed.clone()
xa[:, ::5] = 0.0                                 # exact zeros (padding / ReLU)
qa, sa, za = quantize_asymmetric(xa)
assert qa.dtype == torch.uint8 and float(za) == int(za) and 0 <= int(za) <= 255
x_lo, x_hi = min(xa.min(), 0), max(xa.max(), 0)
assert torch.allclose(sa, (x_hi - x_lo) / 255)
deq = dequantize(qa, sa, za)
assert (deq[:, ::5] == 0).all(), "0.0 must dequantize to exactly 0.0"
assert ((deq - xa).abs() <= sa / 2 + 1e-6).all()
_, s_pos, z_pos = quantize_asymmetric(torch.rand(100) + 2.0)    # all positive: range is [0, max]
assert int(z_pos) == 0, f"all-positive input -> zero point 0, got {z_pos}"
# On non-negative activations asymmetric uses all 256 codes; symmetric wastes the negative half.
q_s, s_s = quantize_symmetric(acts_relu)
q_a, s_a, z_a = quantize_asymmetric(acts_relu)
mse_s = (dequantize(q_s, s_s) - acts_relu).pow(2).mean()
mse_a = (dequantize(q_a, s_a, z_a) - acts_relu).pow(2).mean()
print(f"post-ReLU activations: symmetric MSE {mse_s:.2e}, asymmetric MSE {mse_a:.2e}")
assert mse_a < 0.5 * mse_s
q0, s0, z0 = quantize_asymmetric(torch.zeros(3, 3))
assert torch.isfinite(dequantize(q0, s0, z0)).all()
print("Test 3 passed")

In [ ]:
# Test 4: int8 linear layer == dequantized float matmul, with an exact integer accumulator
x_q, x_s, x_z = quantize_asymmetric(acts_signed)       # (16, 128) uint8
w_q, w_s = quantize_symmetric(W, axis=0)               # (64, 128) int8, (64,) scales
y = int8_linear(x_q, x_s, x_z, w_q, w_s)
assert y.shape == (16, 64) and y.dtype == torch.float32
y_ref = dequantize(x_q, x_s, x_z).double() @ dequantize(w_q, w_s, axis=0).double().T
assert torch.allclose(y.double(), y_ref, rtol=1e-5, atol=1e-5 * y_ref.abs().max()), \
    "int8_linear must equal dequant(x) @ dequant(w).T up to float rounding"
y_fp = acts_signed @ W.T
print(f"relative error of the int8 layer vs the fp32 layer: {(y - y_fp).norm() / y_fp.norm():.2%}")

# Exactness: unit scales expose the accumulator. These dot products exceed 2^24.
g = torch.Generator().manual_seed(1)
xq_big = torch.randint(200, 256, (8, 4096), generator=g).to(torch.uint8)
wq_big = torch.randint(90, 128, (16, 4096), generator=g).to(torch.int8)
exact = (xq_big.long() - 3) @ wq_big.long().T
y_big = int8_linear(xq_big, 1.0, 3, wq_big, torch.ones(16))
assert torch.equal(y_big, exact.to(y_big.dtype)), "the integer accumulation is not exact"
fp32_acc = (xq_big.float() - 3) @ wq_big.float().T
print(f"float32 accumulation would be wrong in {(fp32_acc != exact.float()).sum()} of {exact.numel()} outputs")
print("Test 4 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("int8-quantization", quantize_symmetric=quantize_symmetric,
      quantize_asymmetric=quantize_asymmetric, dequantize=dequantize, int8_linear=int8_linear)

### Why it works

**Error bound.** Inside the clamp range, $q$ is the nearest integer to $x/s + z$, so
$|\,(q - z) - x/s\,| \le \tfrac12$ and $|\hat x - x| \le s/2$. The scales are chosen so nothing
valid gets clamped: symmetric maps $\max|x|$ exactly onto $\pm q_{\max}$; asymmetric maps
$[x_{\min}, x_{\max}]$ onto $[0, 2^b-1]$, and because $z$ is rounded by at most $\tfrac12$ the
endpoints land within half a step of the grid edge, where clamping costs at most $s/2$.

**Exact zero.** With an integer $z$, $x = 0$ gives $q = \operatorname{round}(0) + z = z$ and
$\hat x = (z - z)s = 0$ exactly. That matters because zeros are everywhere — padding, ReLU
outputs, masked positions — and a biased zero adds a systematic offset to every sum.

**Per-channel.** The per-channel step $s_c = \max|x_c|/q_{\max}$ is never larger than the global
one, so its error never is either. One outlier channel inflates a per-tensor scale for *every*
channel: in the setup, nearly half of the normal weights round to zero per-tensor and
their MSE is three orders of magnitude worse than per-channel. Weight scales are
per output channel because they factor out of the dot product: $s_{w,j}$ multiplies the whole
output column $j$. (A per-*input*-channel scale would sit inside $\sum_k$ and break the integer
GEMM — which is exactly why SmoothQuant instead migrates activation outliers into the weights.)

**Integer GEMM.** $\sum_k s_x(q^x_{ik} - z_x)\, s_{w,j} q^w_{jk} = s_x s_{w,j}\sum_k (q^x_{ik}-z_x) q^w_{jk}$:
the scales factor out, so the inner sum is pure integer arithmetic. Each product is at most
$255 \cdot 127 \approx 2^{15}$, so an int32 accumulator is exact for $K < 2^{31}/(255\cdot127) \approx 66\text{k}$.
Float32 has a 24-bit significand, so once a running sum passes $2^{24} \approx 1.7\cdot10^7$ it
can no longer hold every integer — Test 4 shows a float32 matmul getting a sizeable fraction of the
outputs wrong. Real kernels (FBGEMM, cuBLASLt, `torch._int_mm`) instead expand
$\sum_k (q^x - z_x)q^w = \sum_k q^x q^w - z_x \sum_k q^w$ and precompute the weight row sums, so the
zero point costs one subtraction per output.

On CPU in this PyTorch, `int32 @ int32` is exact; int64 or float64 (exact for integers below
$2^{53}$) would also be correct. What is *not* fine is `int8 @ int8` — the result dtype stays int8
and wraps — or `uint8 - z`, which wraps before you ever reach the matmul.

### Complexity

- Quantize / dequantize: $O(n)$ elementwise plus one max-reduction.
- `int8_linear`: $O(MNK)$ integer multiply-adds, $O(MN)$ float multiplies for the rescale.
- Memory: int8 weights are 4× smaller than fp32 and 2× smaller than bf16, plus one scale per channel.

### Common bugs

- Casting with `.to(torch.int8)` without `torch.round` — truncation toward zero doubles the error.
- $q_{\max} = 2^{b-1} = 128$ for symmetric: 128 overflows int8 (or the max element is clamped
  and its error exceeds $s/2$).
- Asymmetric: not widening the range to include 0 (an all-positive input then has a negative
  zero point and 0.0 is not representable), or leaving the zero point as a float.
- Subtracting the zero point from a `uint8` tensor, or `int8 @ int8`, both of which wrap around.
- Accumulating in float32, or dequantizing first and calling it an int8 kernel.
- Per-channel: reducing over the wrong dimension (scale has the wrong shape, or the right shape
  by accident on a square matrix), or broadcasting a `(N,)` weight scale over the rows of `y`
  instead of its columns.
- No guard for an all-zero tensor: scale 0, then 0/0 = NaN.

### Interview follow-ups

1. **Per-tensor vs per-channel vs per-group — what do GPTQ/AWQ use?** Per-tensor is cheapest but
   one outlier sets everyone's step size; per-output-channel fixes that at no GEMM cost because
   the scale factors out of the dot product. At 4 bits even a channel is too coarse, so GPTQ and
   AWQ use per-*group* scales: groups of 128 consecutive input weights share a scale (and zero
   point), and the kernel rescales partial sums group by group.
2. **Why are activations harder to quantize than weights?** Weights are static and roughly
   Gaussian; LLM activations have a few channels with values 10–100× larger than the rest,
   which persist across tokens. Per-tensor activation scales then waste almost all codes.
   LLM.int8() keeps outlier channels in fp16; SmoothQuant divides activations by a per-channel
   factor $s_k$ and multiplies the matching weight rows by $s_k$ (mathematically equivalent),
   moving the difficulty into the easier-to-quantize weights.
3. **Symmetric or asymmetric?** Symmetric (z = 0) keeps the integer GEMM a plain dot product and
   suits roughly zero-centred weights. Asymmetric wastes no codes on one-sided data (post-ReLU,
   post-softmax): about 4× lower MSE in Test 3, at the price of the zero-point correction term
   $z_x \sum_k q^w_{jk}$. Common practice: symmetric weights, asymmetric activations.
4. **Dynamic vs static quantization, and what is calibration?** Dynamic quantization computes
   activation scales at runtime from each batch (or token): accurate, but costs a reduction per
   call. Static quantization fixes activation scales ahead of time from a calibration set,
   using max, percentiles, or MSE/KL-optimal clipping — clipping a few outliers often beats
   covering the full range, since it shrinks the step for everyone else.
5. **FP8 (E4M3 / E5M2) vs INT8?** FP8 has a non-uniform grid: relative precision is constant
   across magnitudes, so it tolerates outliers and wide dynamic ranges better than the uniform
   INT8 grid. E4M3 (more mantissa, max 448) is used for weights and activations, E5M2 (more
   exponent, max 57344) for gradients. Hopper/Blackwell tensor cores run FP8 at the INT8 rate,
   and FP8 still needs per-tensor (or per-block) scaling factors.
6. **Where does the speed-up actually come from?** At small batch (LLM decode), GEMMs are
   memory-bound: int8/int4 weights cut the bytes read per token by 2–4×, which is most of the
   win even if the compute is dequantized to bf16 in registers (weight-only quantization, e.g.
   W4A16). At large batch you are compute-bound, and only true integer/FP8 tensor-core math
   (W8A8) helps.